# Agent Pagination Integrity Benchmark (APIB) v0.1.0

Kaggle / Google Colab analysis notebook. The episode and result files in this release are self-generated synthetic data, not observations from deployed LLM agents.

**Research question:** How often do pagination policies falsely declare a collection complete under empty-but-nonterminal pages, duplicate/omitted records, missing pages, and cursor cycles?

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Works in the repository root, in Kaggle after attaching the APIB dataset,
# or in Colab after cloning the repository.
candidates = [Path('results'), Path('..') / 'results']
try:
    candidates += list(Path('/kaggle/input').glob('**/results'))
    candidates += [p.parent for p in Path('/kaggle/input').glob('**/per_episode_results.csv')]
except Exception:
    pass
try:
    candidates += list(Path('/content').glob('**/results'))
    candidates += [p.parent for p in Path('/content').glob('**/per_episode_results.csv')]
except Exception:
    pass
result_dir = next((p for p in candidates if (p / 'per_episode_results.csv').exists()), None)
if result_dir is None:
    # Colab fallback: clone the public repository.
    import subprocess
    repo_dir = Path('/content/apib-repo')
    if not (repo_dir / 'results' / 'per_episode_results.csv').exists():
        subprocess.run(['git', 'clone', 'https://github.com/MHKhan50/agent-pagination-integrity-benchmark.git', str(repo_dir)], check=True)
    if (repo_dir / 'results' / 'per_episode_results.csv').exists():
        result_dir = repo_dir / 'results'
if result_dir is None:
    raise FileNotFoundError('Attach the APIB Kaggle dataset (containing per_episode_results.csv), or publish/clone the GitHub repository in Colab.')
print('Using:', result_dir.resolve())
runs = pd.read_csv(result_dir / 'per_episode_results.csv')
metrics = pd.read_csv(result_dir / 'policy_metrics.csv')
episodes = pd.read_csv(result_dir / 'episodes.csv') if (result_dir / 'episodes.csv').exists() else None
display(metrics)

## 1. Policy-level results

False completion means the policy reported that the collection was complete even though the unique retrieved record count was below the expected total.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(metrics['policy'], metrics['false_completion_rate'] * 100)
ax.set_ylabel('False-completion rate (%)')
ax.set_title('False completion by pagination policy')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

## 2. Scenario-stratified failure analysis

In [ ]:
scenario_table = runs.groupby(['scenario', 'policy'], as_index=False).agg(
    episodes=('episode_id', 'count'),
    mean_coverage=('coverage', 'mean'),
    false_completion_rate=('false_completion', 'mean'),
    mean_tool_calls=('calls', 'mean')
)
display(scenario_table.sort_values(['scenario', 'policy']))

## 3. Recompute the primary metric from episode-level records

This guards against relying only on a precomputed summary file.

In [ ]:
recomputed = runs.groupby('policy', as_index=False).agg(
    episodes=('episode_id', 'count'),
    mean_coverage=('coverage', 'mean'),
    false_completion_rate=('false_completion', 'mean'),
    mean_tool_calls=('calls', 'mean')
)
display(recomputed)

## Interpretation and limitations

The benchmark is a deterministic synthetic policy simulator. It does not measure a deployed LLM, does not include live API calls, and assumes that the expected collection size is available to the verification policy. The count-verified strategy can detect a mismatch; it cannot recover a missing page by itself. Production studies should test real API adapters, expiring tokens, rate limits, concurrent data mutation, and LLM-driven tool selection.

## References
- Google Ad Manager API pagination: https://developers.google.com/ad-manager/api/beta/pagination
- Microsoft Graph paging: https://learn.microsoft.com/en-us/graph/paging
